In [98]:
##Code in this file has been modified from Boris Murmann's github below, and the open-source textbook from Harald Pretl and colleagues:
# https://github.com/bmurmann/Book-on-gm-ID-design/blob/main/starter_files_open_source_tools/gf180mcuD/techsweep_plots_from_mat.ipynb
# Murmann and his colleague have written a book on gm/id-based design, "Systematic Design of Analog CMOS Circuits" (2017)

#Pretl's Open-Source Book: https://iic-jku.github.io/analog-circuit-design/analog_circuit_design.pdf

# Copyright 2024 Harald Pretl
# Licensed under the Apache License, Version 2.0 (the “License”); you may not use this
# file except in compliance with the License. You may obtain a copy of the License at
# http://www.apache.org/licenses/LICENSE-2.0


import numpy as np
import scipy.constants as sc
import matplotlib.pyplot as plt
from pygmid import Lookup as lk

nfet = lk('../nfet_03v3.mat')
pfet = lk('../pfet_03v3.mat')
VDS1 = 1.65

In [ ]:
# define the given parameters as taken from the specification table or inital guesses
c_load = 35e-12 #Expected given parasitics + breadboard mounting of device
gm_id_m12 = 9
gm_id_m34 = 8 
gm_id_m569 = 4
gm_id_m78 = 6
#length in microns
l_m12 = 1       #gain transistors in diff pair
l_m34 = 0.8     #cascode for diff pair
l_m569 = 1      #PMOS active load (gain-limiting) and CS amp... chosen to be the same to simplify balance condition
l_m78 = 1        #ISS + NMOS current source for CS

# Current Source Limitations
i_ss_tot = 70e-6               #Reference current is 10uA. Note that if some internal design uses something like 30uA, you can mirror that value too
i_do = 120e-6
i_casb = 10e-6              #bias current for cascode bias voltage generation
i_ss = i_ss_tot-i_casb
#Target Specifications
f_gb = 2e6 # -3dB bandwidth of the voltage buffer
av = 2000

In [100]:
c_comp = 0.22*c_load #good rule of thumb
print("Miller Cap = ", round(c_comp*10**12,3), "pF")
slew_r = i_ss/c_comp
print("Estimated slew rate", round(slew_r/10**6,3), "V/us")

gm_m12 = gm_id_m12 * (i_ss/2)
gm_m34 = gm_id_m34 * (i_ss/2)
print("Target gm_m12", round(gm_m12*10**3,3), "ms")
print("Target gm_m34", round(gm_m34*10**3,3), "ms")

gbw = gm_m12/c_comp
print(round(gbw/10**6), "MHz")

#try to place non-dominant pole 10x above target f_gb
ro_m34 = (10*f_gb)*2*np.pi*c_load/(gm_m34*gm_m12)
print("Minimum output impedance from m56 =", round(ro_m34/10**3, 3), "kOhm") #this is coming out unusually small IMO



Miller Cap =  7.7 pF
Estimated slew rate 7.792 V/us
Target gm_m12 0.27 ms
Target gm_m34 0.24 ms
35 MHz
Minimum output impedance from m56 = 67.874 kOhm


In [110]:
gm_m569 = gm_id_m569 * (i_ss/2)
#estimate stage 1 gain

#Intrinsic gain of mirrors and diff pair
gm_gds_m12 = nfet.lookup('GM_GDS', GM_ID=gm_id_m12, L=l_m12, VDS=VDS1, VSB=0)
gm_gds_m34 = nfet.lookup('GM_GDS', GM_ID=gm_id_m34, L=l_m34, VDS=VDS1, VSB=0)
gm_gds_m569 = pfet.lookup('GM_GDS', GM_ID=gm_id_m569, L=l_m569, VDS=VDS1, VSB=0)


#get diff pair gds values
ro_m12 = gm_gds_m12 / gm_m12
ro_m34 = gm_gds_m34 / gm_m34 

ro_cascode = (gm_m12*ro_m12+1)*ro_m34+ro_m12
print("ro_cascode =", round(ro_cascode /1000,2),"kOhm"  )

#get current mirror ro
gm_m56 = gm_id_m569 * (i_ss/2)
print("gm_m569 =", round(gm_m56/1e-3, 4), 'mS')
ro_m56 = gm_gds_m569 / gm_m56

print("ro_m56 =", round(ro_m56/1000,2), 'kOhm')
a0 = gm_m12 * (ro_cascode*ro_m56)/(ro_cascode + ro_m56)
print('a0 =', round(20*np.log10(a0), 1), 'dB')

#zero position
gm_m9 = gm_id_m569 * i_do
f_zero = gm_m34*ro_m56*(gm_m9/c_comp)/(2*np.pi)
f_nondom = gm_m34*gm_m9*ro_m56/(2*np.pi*c_load)
print("f_zero =", round(f_zero/10**6, 3), 'MHz')        #this seems not correct
print("f_nondom =", round(f_nondom/10**6, 3), 'MHz')    #this also seems not correct


gm_cgs_m56 = pfet.lookup('GM_CGS', GM_ID=gm_id_m569, L=l_m569, VDS=VDS1, VSB=0)
gm_cdd_m56 = pfet.lookup('GM_CDD', GM_ID=gm_id_m569, L=l_m569, VDS=VDS1, VSB=0)
gm_cdd_m34 = nfet.lookup('GM_CDD', GM_ID=gm_id_m34, L=l_m34, VDS=VDS1, VSB=0)
c_load_parasitic = abs(gm_m56/gm_cgs_m56) + abs(gm_m56/gm_cdd_m56) + abs(gm_m34/gm_cdd_m34)

f_mirror = gm_m56/(2*np.pi*c_load_parasitic)
print("f_mirror,est =", round(f_mirror/10**6, 3), 'MHz')


ro_cascode = 485805.14 kOhm
gm_m569 = 0.12 mS
ro_m56 = 2330.02 kOhm
a0 = 55.9 dB
f_zero = 5548.064 MHz
f_nondom = 1220.574 MHz
f_mirror,est = 495.133 MHz


In [102]:
vgs_m12 = nfet.look_upVGS(GM_ID=gm_id_m12, L=l_m12, VDS=VDS1, VSB=0.0)
vgs_m34 = nfet.look_upVGS(GM_ID=gm_id_m34, L=l_m34, VDS=VDS1, VSB=0.0)
vgs_m78 = nfet.look_upVGS(GM_ID=gm_id_m78, L=l_m78, VDS=VDS1, VSB=0.0)
vgs_m569 = pfet.look_upVGS(GM_ID=gm_id_m569, L=l_m569, VDS=VDS1, VSB=0.0)

print("vgs_m34 = ", round(1*vgs_m34,3))

# calculate all widths
id_w_m12 = nfet.lookup('ID_W', GM_ID=gm_id_m12, L=l_m12, VDS=vgs_m12, VSB=0)
w_m12 = (i_ss/2) / id_w_m12
w_m12_round = max(round(w_m12*2)/2, 0.5)
print('M1/2 W =', round(w_m12, 2), 'um, rounded W =', w_m12_round, 'um')

#M34
id_w_m34 = nfet.lookup('ID_W', GM_ID=gm_id_m34, L=l_m34, VDS=vgs_m34, VSB=0)
w_m34 = (i_ss/2) / id_w_m34
w_m34_round = max(round(w_m34*2)/2, 0.5)
print('M3/4 W =', round(w_m34, 2), 'um, rounded W =', w_m34_round, 'um')

#M56,9
id_w_m569 = pfet.lookup('ID_W', GM_ID=gm_id_m569, L=l_m569, VDS=vgs_m569, VSB=0)
w_m56 = (i_ss/2) / id_w_m569
w_m56_round = max(round(w_m56*2)/2, 0.5)
print('M5/6 W =', round(w_m56, 2), 'um, rounded W =', w_m56_round, 'um')

w_m9 = i_do / id_w_m569
w_m9_round = max(round(w_m9*2)/2, 0.5)
print('M9 W =', round(w_m9, 2), 'um, rounded W =', w_m9_round, 'um')


#M7
id_w_m7 = nfet.look_up('ID_W', GM_ID=gm_id_m78, L=l_m78, VDS=vgs_m78, VSB=0)
w_m7 = i_do / id_w_m7
w_m7_round = max(round(w_m7*2)/2, 0.5)
print('M7 W =', round(w_m7, 2), 'um, rounded W =', w_m7_round, 'um')

#m8
w_m8 = (w_m7/2) * w_m9/w_m56
w_m8_round = max(round(w_m8*2)/2, 0.5)
print('M8 W =', round(w_m8, 2), 'um, rounded W =', w_m8_round, 'um')


vgs_m34 =  0.892
M1/2 W = 10.64 um, rounded W = 10.5 um
M3/4 W = 6.61 um, rounded W = 6.5 um
M5/6 W = 8.94 um, rounded W = 9.0 um
M9 W = 35.76 um, rounded W = 36.0 um
M7 W = 18.23 um, rounded W = 18.0 um
M8 W = 36.47 um, rounded W = 36.5 um


In [103]:
gm_gds_m78 = nfet.lookup('GM_GDS', GM_ID=gm_id_m78, L=l_m78, VDS=VDS1, VSB=0)
gm_m8 = gm_id_m78 * i_do

gds_m78 = gm_m8/gm_gds_m78
gds_m9 = gm_m9/gm_gds_m569

a1 = gm_m9/(gds_m9 + gds_m78)
print('a1 =', round(20*np.log10(a1), 1), 'dB')
av = a1*a0
print('av =', round(20*np.log10(av), 1), 'dB')

ro_8 = 1/gds_m78 
fp1 = 1/(2*np.pi*ro_8*(c_comp+c_load)+c_comp/gm_m34+gm_m9*ro_8*ro_m569*c_load)
print("Dominant Pole frequency", round(fp1,1), "Hz")

a1 = 41.8 dB
av = 97.8 dB
Dominant Pole frequency 54.9 Hz
